In [5]:
import pycave
from pycave.clustering import KMeans
import numpy as np
import torch

ModuleNotFoundError: No module named 'pycave'

In [ ]:
import glob
directory = "../output/xenium/fullPanel/graphs/3NB/xenium_subgraph3NB5000_aug2023/dmax3.0/3.0/min10/"
pattern = "*_node_meta.csv"  # Replace with your desired pattern
files = glob.glob(f"{directory}/{pattern}")

In [ ]:
files

In [ ]:
len(files)

In [ ]:
sample_names = [f_name.split("/")[-1] for f_name in files]

In [ ]:
tmas = [f_name.split("_")[1] for f_name in sample_names]

sample_names = [f_name.split("_")[0] for f_name in sample_names]

In [ ]:
tmas + sample_names

In [ ]:
X_list = []
npy_files = [x +"_50_embeddings_" +y +".npy" for x,y in zip(sample_names,tmas)]

In [ ]:
npy_files 

In [ ]:
npy_dir = directory
X_list = [np.load(npy_dir+npy) for npy in npy_files]

In [ ]:
[x.shape for x in X_list]

In [ ]:
X = np.concatenate(X_list)

In [ ]:
X.shape

In [ ]:
sample_size = [x.shape[0] for x in X_list]
sample_size

In [ ]:
from  pycave.bayes import GaussianMixture
import random
#import matplotlib.pyplot as plt
import pathlib

In [ ]:
print(torch.cuda.get_arch_list())

In [ ]:
print(torch.cuda.current_device())

In [ ]:
k_list = [7,8,9,10,11,12,13]
estimator_gmms = [GaussianMixture(x, trainer_params=dict(accelerator='gpu', devices=1,
                                                       enable_progress_bar = True),
                                init_strategy = "kmeans++", batch_size = 15000000) for x in k_list]
for estimator_gmm in estimator_gmms:
    estimator_gmm.fit(torch.from_numpy(X))
cluster_id_gmms = [estimator_gmm.predict(torch.from_numpy(X)) for estimator_gmm in estimator_gmms]
for k in k_list: 
    list_i = k - min(k_list)
    start_i = 0
    sample_cluster_gmm = list()
    out_dir = npy_dir+"gmm"+str(k)+"_5k_trained/"
    pathlib.Path(out_dir).mkdir(parents=True, exist_ok=True)

    for s_size, region in zip(sample_size, npy_files):
        sample_cluster_gmm.append( cluster_id_gmms[list_i][start_i:(start_i+s_size)])
        start_i = start_i+s_size

    for c_id, region in zip(sample_cluster_gmm, npy_files):
        f_name = region.split("_50_embeddings")[0]
        t_name = region.split("_")[3].split(".npy")[0]
        np.savetxt(out_dir+f_name+'_noMeans_'+t_name+'.txt', c_id.numpy().astype(int), fmt="%s")